# Install the optics correctors (trims) for the response matrix (LEP3 Z, V11f)

LEP3 equivalent of the corrector trims installed in `FCCee_LCC_Z_V107p0p1_imperfections_model/01_install_correctors.py`
(notebook 01 of LEP3 installs only the markers, BPMs and orbit correctors).

- Normal quadrupole trims (`knl1_*`) at all normal (powered) quadrupoles, skew quadrupole trims (`ksl1_*`) at all normal (powered) sextupoles.
- One knob per magnet: the slices of the same magnet (e.g. `2*SF1A`, `4*SCRABL`, `QX2L,IMAG4,QX2L`, the two `QD1AM` halves) share the knob,
  split according to their length, as for the imperfections (notebook 02). The knob is named after the first slice, e.g. `knl1_qd1am:15`.
- The line is cycled to `rf800:0`, as the lines with imperfections (notebook 02). The response matrices are built on this line,
  so the phases need no re-referencing in the corrections (`RM_PHASE_ORIGIN = 'rf800:0'`).

The saved line is the reference line of the LEP3 model: it is the input of notebook 02 and of the scripts 02-04 of this folder.

In [1]:
import numpy as np
import xtrack as xt

In [2]:
# Load line (from notebook 01)
line_version = 'LEP3_Z_11f'
line = xt.Line.from_json('../lattices/reference_lattice/LEP3_Z_11f_wMarkers_wBPMs_wCorr.json')
line.cycle(name_first_element='rf800:0', inplace=True) # same start as the lines with imperfections (notebook 02)
line.configure_radiation(model=None, model_beamstrahlung=None)

tw0 = line.twiss4d()
print(tw0.qx, tw0.qy)
print(tw0.dqx, tw0.dqy)
tt = line.get_table()

Loading line from dict:   0%|          | 0/14064 [00:00<?, ?it/s]

/Users/prebibaj/miniforge3/envs/fcc-2026/lib/python3.13/site-packages/xtrack/beam_elements/cavity.py:124: FutureWarning: `lag` (in degrees) is deprecated and will be removed in a future version. Please use `phase` (in radians) instead. If you see this warning while loading a saved line from a previous version of Xsuite, please regenerate the line with the current version to use phase instead of lag. Note that if both `lag` and `phase` are set, the effect is the sum of the two, with `lag` converted to radians.  This deprecation is part of the interface cleanup in view of the 1.0 release.
  self.lag = lag


Done loading line from dict.           
195.16626417036852 87.21079538065459
3.2907502890115197 2.2972873556173257


### Inspect elements

In [3]:
# Dipoles (combined-function arc dipoles are Bend, FF dipoles are RBend)
ttbend = tt.rows[np.isin(tt.element_type, ['Bend', 'RBend'])]
print('There are', len(ttbend), 'bends in the lattice, of which', sum(ttbend.element_type == 'Bend'), 'are Bend and', sum(ttbend.element_type == 'RBend'), 'are RBend.')

There are 1766 bends in the lattice, of which 1696 are Bend and 70 are RBend.


In [4]:
# Quadrupoles
ttquad = tt.rows[tt.element_type=='Quadrupole']
mask = [quad for quad in ttquad.name if line.element_dict[quad].k1s==0 and abs(line.element_dict[quad].k1)>0]
ttquadno = ttquad.rows[np.isin(ttquad.name, mask)]
mask = [quad for quad in ttquad.name if abs(line.element_dict[quad].k1s)>0 and line.element_dict[quad].k1==0]
ttquadsk = ttquad.rows[np.isin(ttquad.name, mask)]
mask = [quad for quad in ttquad.name if abs(line.element_dict[quad].k1s)==0 and line.element_dict[quad].k1==0]
ttquadzero = ttquad.rows[np.isin(ttquad.name, mask)]
print('There are', len(ttquad), 'quads in the lattice, of which', len(ttquadno), 'are normal,', len(ttquadsk), 'are skew, and', len(ttquadzero), 'have zero strength.')

There are 1872 quads in the lattice, of which 1872 are normal, 0 are skew, and 0 have zero strength.


In [5]:
# Sextupoles
ttsext = tt.rows[tt.element_type=='Sextupole']
mask = [sext for sext in ttsext.name if line.element_dict[sext].k2s==0 and abs(line.element_dict[sext].k2)>0]
ttsextno = ttsext.rows[np.isin(ttsext.name, mask)]
mask = [sext for sext in ttsext.name if abs(line.element_dict[sext].k2s)>0 and line.element_dict[sext].k2==0]
ttsextsk = ttsext.rows[np.isin(ttsext.name, mask)]
mask = [sext for sext in ttsext.name if abs(line.element_dict[sext].k2s)==0 and line.element_dict[sext].k2==0]
ttsextzero = ttsext.rows[np.isin(ttsext.name, mask)]
print('There are', len(ttsext), 'sexts in the lattice, of which', len(ttsextno), 'are normal,', len(ttsextsk), 'are skew, and', len(ttsextzero), 'have zero strength.')

There are 696 sexts in the lattice, of which 696 are normal, 0 are skew, and 0 have zero strength.


In [6]:
# Instruments
ttbpm = tt.rows['bpm.*']
print('There are', len(ttbpm), 'BPMs in the lattice, almost for every quadrupole.')

There are 1860 BPMs in the lattice, almost for every quadrupole.


In [7]:
# Orbit correctors
ttcorr = tt.rows['.*cor.*']
ttcorrh = tt.rows['.*hcor.*']
ttcorrv = tt.rows['.*vcor.*']
print('There are', len(ttcorr), 'correctors in the lattice, of which', len(ttcorrh), 'are horizontal and', len(ttcorrv), 'are vertical.')

There are 3744 correctors in the lattice, of which 1872 are horizontal and 1872 are vertical.


### Add optics correctors

In [8]:
def group_magnet_slices(tt, element_names, tol=1e-6):
    """
    Group consecutive slices of the same magnet: same family name (name without ':<n>') and touching (s_end = next s_start).
    Returns a list of magnets, each one being the list of its slice names (in line order). Same as in notebook 02.
    """
    tt_sel = tt.rows[np.isin(tt.name, element_names)]
    family = lambda name: name.split(':')[0]
    magnets = []
    for name, s_start, s_end in zip(tt_sel.name, tt_sel.s_start, tt_sel.s_end):
        if magnets and family(name) == family(magnets[-1][-1]) and abs(s_start - previous_s_end) < tol:
            magnets[-1].append(name)
        else:
            magnets.append([name])
        previous_s_end = s_end
    return magnets


def add_correctors_per_magnet(line, magnets, type='normal', order=1, switch_name=None):
    """
    Same as helpers_for_imperfections_model.add_correctors, but with one knob per magnet (named after its first slice),
    shared by all its slices in proportion to their length, so that the knob is the integrated strength of the magnet.
    Returns the list of knob names.
    """
    if switch_name is not None:
        line.vars[switch_name] = 1

    if type == 'normal':
        attr = 'knl'
    elif type == 'skew':
        attr = 'ksl'
    else:
        raise ValueError(f"Unsupported corrector type '{type}'. Supported types are 'normal' and 'skew'.")

    knob_names = []
    for slices in magnets:
        knob_name = attr + str(order) + '_' + slices[0]
        line.vars[knob_name] = 0
        knob_names.append(knob_name)
        total_length = sum(line[name].length for name in slices)
        for name in slices:
            current = getattr(line.element_refs[name], attr)[order]._expr # <-- Take current expression
            if current is None: # <-- to handle the case in which the attribute has no expression, just a value
                current = getattr(line.element_refs[name], attr)[order]._value
            increment = line.vars[knob_name] * (line[name].length / total_length)
            getattr(line.element_refs[name], attr)[order] = (current + increment * line.vars[switch_name]) if switch_name is not None else (current + increment)
    return knob_names

In [9]:
quad_magnets = group_magnet_slices(tt, ttquadno.name)
sext_magnets = group_magnet_slices(tt, ttsextno.name)
print(f'{len(ttquadno)} normal quadrupole slices -> {len(quad_magnets)} magnets.')
print(f'{len(ttsextno)} normal sextupole slices -> {len(sext_magnets)} magnets.')

# Normal optics corrector trims at all normal (powered) quadrupoles
qno_knobs = add_correctors_per_magnet(line, quad_magnets, type='normal', order=1, switch_name='on_qno_corrector')
# Skew optics corrector trims at all normal (powered) sextupoles
qsk_knobs = add_correctors_per_magnet(line, sext_magnets, type='skew', order=1, switch_name='on_qsk_corrector')

1872 normal quadrupole slices -> 1782 magnets.
696 normal sextupole slices -> 484 magnets.


In [10]:
# Save the line with installed optics correctors (reference line of the LEP3 model)
line.to_json(f'../lattices/reference_lattice/{line_version}_wMarkers_wBPMs_wCorr_wTrims.json')

### Check the corrector installation

In [11]:
# A magnet split in two slices (the two QD1AM halves at the cell boundary)
split_magnet = [magnet for magnet in quad_magnets if magnet[0].startswith('qd1am')][0]
for name in split_magnet:
    line.element_refs[name].knl[1]._info()

#  element_refs['qd1am:15'].knl[1]._get_value()
   element_refs['qd1am:15'].knl[1] = 0.0

#  element_refs['qd1am:15'].knl[1]._expr
   element_refs['qd1am:15'].knl[1] = (0.0 + ((vars['knl1_qd1am:15'] * 0.5) * vars['on_qno_corrector']))

#  element_refs['qd1am:15'].knl[1]._expr._get_dependencies()
   vars['knl1_qd1am:15'] = 0
   vars['on_qno_corrector'] = 1

#  element_refs['qd1am:15'].knl[1] does not influence any target

#  element_refs['qd1am:16'].knl[1]._get_value()
   element_refs['qd1am:16'].knl[1] = 0.0

#  element_refs['qd1am:16'].knl[1]._expr
   element_refs['qd1am:16'].knl[1] = (0.0 + ((vars['knl1_qd1am:15'] * 0.5) * vars['on_qno_corrector']))

#  element_refs['qd1am:16'].knl[1]._expr._get_dependencies()
   vars['knl1_qd1am:15'] = 0
   vars['on_qno_corrector'] = 1

#  element_refs['qd1am:16'].knl[1] does not influence any target



In [12]:
# A sextupole split in four slices (crab sextupole)
split_magnet = [magnet for magnet in sext_magnets if magnet[0].startswith('scrab')][0]
for name in split_magnet:
    line.element_refs[name].ksl[1]._info()

#  element_refs['scrabl'].ksl[1]._get_value()
   element_refs['scrabl'].ksl[1] = 0.0

#  element_refs['scrabl'].ksl[1]._expr
   element_refs['scrabl'].ksl[1] = (0.0 + ((vars['ksl1_scrabl'] * 0.25) * vars['on_qsk_corrector']))

#  element_refs['scrabl'].ksl[1]._expr._get_dependencies()
   vars['ksl1_scrabl'] = 0
   vars['on_qsk_corrector'] = 1

#  element_refs['scrabl'].ksl[1] does not influence any target

#  element_refs['scrabl:0'].ksl[1]._get_value()
   element_refs['scrabl:0'].ksl[1] = 0.0

#  element_refs['scrabl:0'].ksl[1]._expr
   element_refs['scrabl:0'].ksl[1] = (0.0 + ((vars['ksl1_scrabl'] * 0.25) * vars['on_qsk_corrector']))

#  element_refs['scrabl:0'].ksl[1]._expr._get_dependencies()
   vars['ksl1_scrabl'] = 0
   vars['on_qsk_corrector'] = 1

#  element_refs['scrabl:0'].ksl[1] does not influence any target

#  element_refs['scrabl:1'].ksl[1]._get_value()
   element_refs['scrabl:1'].ksl[1] = 0.0

#  element_refs['scrabl:1'].ksl[1]._expr
   element_refs['scrabl:1'].ksl[1] = 

In [13]:
line.vars['on_qno_corrector'] = 1
line.vars['on_qsk_corrector'] = 1

In [14]:
vartable = line.vars.get_table()

In [15]:
qno_vars = vartable.rows['knl1_.*']
print(len(qno_vars), list(qno_vars.name))

1782 ['knl1_qd5f:0', 'knl1_qf4f:0', 'knl1_qd3f:0', 'knl1_qf2f:0', 'knl1_qd1f:0', 'knl1_qf2a:35', 'knl1_qf2a:36', 'knl1_qd2a:33', 'knl1_qf1a:33', 'knl1_qd1a:35', 'knl1_qf3a:26', 'knl1_qd1a:36', 'knl1_qf3a:27', 'knl1_qd1a:37', 'knl1_qf3a:28', 'knl1_qd1a:38', 'knl1_qf1a:34', 'knl1_qd2a:34', 'knl1_qf2a:37', 'knl1_qf2a:38', 'knl1_qd2a:35', 'knl1_qf1a:35', 'knl1_qd1am:15', 'knl1_qf1a:36', 'knl1_qd2a:36', 'knl1_qf2a:39', 'knl1_qf2a:40', 'knl1_qd2a:37', 'knl1_qf1a:37', 'knl1_qd1a:39', 'knl1_qf3a:29', 'knl1_qd1a:40', 'knl1_qf3a:30', 'knl1_qd1a:41', 'knl1_qf3a:31', 'knl1_qd1a:42', 'knl1_qf1a:38', 'knl1_qd2a:38', 'knl1_qf2a:41', 'knl1_qf2a:42', 'knl1_qd2a:39', 'knl1_qf1a:39', 'knl1_qd1am:17', 'knl1_qf1a:40', 'knl1_qd2a:40', 'knl1_qf2a:43', 'knl1_qf2a:44', 'knl1_qd2a:41', 'knl1_qf1a:41', 'knl1_qd1a:43', 'knl1_qf3a:32', 'knl1_qd1a:44', 'knl1_qf3a:33', 'knl1_qd1a:45', 'knl1_qf3a:34', 'knl1_qd1a:46', 'knl1_qf1a:42', 'knl1_qd2a:42', 'knl1_qf2a:45', 'knl1_qf2a:46', 'knl1_qd2a:43', 'knl1_qf1a:43', 'knl1

In [16]:
qsk_vars = vartable.rows['ksl1_.*']
print(len(qsk_vars), list(qsk_vars.name))

484 ['ksl1_sd1a:13', 'ksl1_sf1a:27', 'ksl1_sd2a:13', 'ksl1_sd2a:14', 'ksl1_sf1a:29', 'ksl1_sd1a:14', 'ksl1_sd1a:15', 'ksl1_sf1a:31', 'ksl1_sd2a:15', 'ksl1_sd2a:16', 'ksl1_sf1a:33', 'ksl1_sd1a:16', 'ksl1_sd1a:17', 'ksl1_sf1a:35', 'ksl1_sd2a:17', 'ksl1_sd2a:18', 'ksl1_sf1a:37', 'ksl1_sd1a:18', 'ksl1_sd1a:19', 'ksl1_sf1a:39', 'ksl1_sd2a:19', 'ksl1_sd2a:20', 'ksl1_sf1a:41', 'ksl1_sd1a:20', 'ksl1_sd1a:21', 'ksl1_sf1a:43', 'ksl1_sd2a:21', 'ksl1_sd2a:22', 'ksl1_sf1a:45', 'ksl1_sd1a:22', 'ksl1_sd1a:23', 'ksl1_sf1a:47', 'ksl1_sd2a:23', 'ksl1_sd2a:24', 'ksl1_sf1a:49', 'ksl1_sd1a:24', 'ksl1_sd1a:25', 'ksl1_sf1a:51', 'ksl1_sd2a:25', 'ksl1_sd2a:26', 'ksl1_sf1a:53', 'ksl1_sd1a:26', 'ksl1_sd1a:27', 'ksl1_sf1a:55', 'ksl1_sd2a:27', 'ksl1_sd2a:28', 'ksl1_sf1a:57', 'ksl1_sd1a:28', 'ksl1_sd1a:29', 'ksl1_sf1a:59', 'ksl1_sd2a:29', 'ksl1_sd2a:30', 'ksl1_sf1a:61', 'ksl1_sd1a:30', 'ksl1_sd1a:31', 'ksl1_sf1a:63', 'ksl1_sd2a:31', 'ksl1_sd2a:32', 'ksl1_sf1a:65', 'ksl1_sd1a:32', 'ksl1_sd1a:33', 'ksl1_sf1a:67', 'ks

In [17]:
# Check: the knob moves the integrated strength of the whole magnet by its value
tw_before = line.twiss4d()
line.vars[qno_knobs[0]] = 1e-5
tw_after = line.twiss4d()
line.vars[qno_knobs[0]] = 0
print(f'{qno_knobs[0]}: dqx = {tw_after.qx - tw_before.qx:.3e}, dqy = {tw_after.qy - tw_before.qy:.3e}')

knl1_qd5f:0: dqx = 2.317e-05, dqy = -7.969e-05
